#### 0. Prerequisites

In [1]:
from abc import ABC, abstractmethod
from typing import List, Tuple, Dict, Set
import numpy as np
import pandas as pd
import ir_datasets

In [2]:
_dataset = ir_datasets.load("beir/nfcorpus/dev")
_dataset

Dataset(id='beir/nfcorpus/dev', provides=['docs', 'queries', 'qrels'])

In [3]:
_corpus = {}
for doc in _dataset.docs_iter():
    passage = f"{doc.title or ''} {doc.text or ''}".strip()
    _corpus[doc.doc_id] = passage

_queries = {q.query_id: q.text for q in _dataset.queries_iter()}

_relevant_docs = {}
for qrel in _dataset.qrels_iter():
    if qrel.relevance > 0:
        if qrel.query_id not in _relevant_docs:
            _relevant_docs[qrel.query_id] = set()
        _relevant_docs[qrel.query_id].add(qrel.doc_id)

In [4]:
# base class for all retrievers
class BaseRetriever(ABC):
    @abstractmethod
    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        """return list of (doc_id, score) for top-k results"""
        pass


    @abstractmethod
    def get_name(self) -> str:
        pass

In [5]:
# evaluation helper functions
def recall_at_k(results, gold_set, k):
    retrieved = [doc_id for doc_id, _ in results[:k]]
    return len(set(retrieved) & set(gold_set)) / len(gold_set)

def hit_at_k(results, gold_set, k):
    retrieved = [doc_id for doc_id, _ in results[:k]]
    return int(len(set(retrieved) & set(gold_set)) > 0)

def reciprocal_rank(results, gold_set) -> float:
    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in gold_set:
            return 1.0 / rank
    return 0.0

def dcg_at_k(relevant_binary: List[int], k: int) -> float:
    dcg = 0.0
    for i in range(min(k, len(relevant_binary))):
        discount = 1.0 / np.log2(i + 2)
        dcg += relevant_binary[i] * discount
    return dcg

def ndcg_at_k(retrieved_ids: List[str], gold_set: Set[str], k: int) -> float:
    relevant_binary = [1 if doc_id in gold_set else 0 for doc_id in retrieved_ids[:k]]
    dcg = dcg_at_k(relevant_binary, k)

    ideal_count = min(len(gold_set), k)
    ideal_relevant = [1] * ideal_count + [0] * (k - ideal_count)
    idcg = dcg_at_k(ideal_relevant, k)

    return dcg / idcg if idcg > 0 else 0.0

In [6]:
# evaluation function
def evaluate_retriever(
        retriever: BaseRetriever,
        queries: Dict[str, str],
        relevant_docs: Dict[str, Set[str]],
        ks=None
) -> Dict:
    if ks is None:
        ks = [1, 5, 10, 50]

    results = {f"recall@{k}": [] for k in ks}
    results.update({f"hit@{k}": [] for k in ks})
    results.update({f"ndcg@{k}": [] for k in ks})
    results["mrr"] = []

    for qid, qtext in queries.items():
        if qid not in relevant_docs:
            continue

        gold_set = relevant_docs[qid]
        retrieved = retriever.search(qtext, k=max(ks))

        for k in ks:
            results[f"recall@{k}"].append(recall_at_k(retrieved, gold_set, k))
            results[f"hit@{k}"].append(hit_at_k(retrieved, gold_set, k))
            results[f"ndcg@{k}"].append(ndcg_at_k(retrieved_ids, gold_set, k))

        results["mrr"].append(reciprocal_rank(retrieved, gold_set))

    return {metric: np.mean(values) for metric, values in results.items()}

###### 0.1 Additional Preprocessing

In [7]:
# data visualization
docs_df = pd.DataFrame(
    {
        "doc_id": d.doc_id,
        "title": d.title,
        "text": d.text,
        "url": d.url
    }
    for d in _dataset.docs_iter()
)
print(docs_df.shape)
#docs_df.head()

queries_df = pd.DataFrame(
    {
        "query_id": q.query_id,
        "text": q.text,
        "url": q.url,
    }
    for q in _dataset.queries_iter()
)
print(queries_df.shape)
#queries_df.head()

qrels_df = pd.DataFrame(
    {
        "query_id": r.query_id,
        "doc_id": r.doc_id,
        "relevance": r.relevance,
        "iteration": r.iteration,
    }
    for r in _dataset.qrels_iter()
)
print(qrels_df.shape)
#qrels_df.head()

(3633, 4)
(324, 3)
(11385, 4)


In [14]:
# average document length
lengths = np.array([len(text.split()) for text in _corpus.values()])

print(f"Average words/doc: {lengths.mean():.2f}")
print(f"Median words/doc: {np.median(lengths):.2f}")
print(f"25th percentile: {np.percentile(lengths, 25):.2f}")
print(f"75th percentile: {np.percentile(lengths, 75):.2f}")
print(f"Min: {lengths.min()}")
print(f"Max: {lengths.max()}")

Average words/doc: 233.77
Median words/doc: 237.00
25th percentile: 185.00
75th percentile: 273.00
Min: 17
Max: 1481


**Note:** Document length is moderate at most so chunking will be saved for our final corpus/retriever----doing so ended up hurting results for this dataset.

#### 1. Sparse Retriever: Keyword Search with BM25

In [16]:
from rank_bm25 import BM25Okapi

from nltk.stem import PorterStemmer
import re
import unicodedata

In [17]:
stemmer = PorterStemmer()

GREEK_MAP = {
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "δ": "delta",
    "κ": "kappa",
    "μ": "mu",
}

def normalize_text(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = (text.lower()
        .replace("-", " ")
        .replace("/", " ")
        .replace("&", " and ")
    )

    # remove punctuation but keep letters/numbers/spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    for symbol, name in GREEK_MAP.items():
        text = text.replace(symbol, f" {name} ")

    return text

In [18]:
def tokenize(text: str, use_stemming: bool = True) -> List[str]:
    text = normalize_text(text)
    tokens = text.split()
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

In [19]:
class BM25Retriever(BaseRetriever):
    def __init__(self, corpus: Dict[str, str], use_stemming: bool = True):
        self.corpus = corpus
        self.doc_ids: list[str] = list(corpus.keys())
        self.doc_texts = [corpus[doc_id] for doc_id in self.doc_ids]

        tokenized_docs = [tokenize(text, use_stemming) for text in self.doc_texts]
        self.bm25 = BM25Okapi(tokenized_docs)
        self.use_stemming = use_stemming


    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        tokenized_query = tokenize(query, use_stemming=self.use_stemming)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:k]

        results = []
        for idx in top_indices:
            doc_id = self.doc_ids[idx]
            score = float(scores[idx])
            results.append((doc_id, score))

        return results


    def get_name(self) -> str:
        return "BM25"

In [20]:
bm25 = BM25Retriever(_corpus, use_stemming=True)
bm25_scores = evaluate_retriever(bm25, _queries, _relevant_docs)
for metric, value in bm25_scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0361
recall@5: 0.0871
recall@10: 0.1300
recall@50: 0.2010
hit@1: 0.4136
hit@5: 0.5895
hit@10: 0.6543
hit@50: 0.7407
mrr: 0.4941


In [27]:
failed = []

for qid, qtext in _queries.items():
    if qid not in _relevant_docs:
        continue

    gold = _relevant_docs[qid]
    bm25_results = bm25.search(qtext)
    retrieved_ids = [doc_id for doc_id, _ in bm25_results]

    if len(set(retrieved_ids) & set(gold)) == 0:
        failed.append((qid, qtext, list(gold)[:3], bm25_results[:3]))

failed[:1]

[('PLAIN-22',
  'Why are Cancer Rates so Low in India?',
  ['MED-5333', 'MED-2608', 'MED-5329'],
  [('MED-2514', 17.619953974531818),
   ('MED-3000', 15.958485292981862),
   ('MED-2217', 15.69993944802342)])]

**Notes**
* Recall@k: fraction of all relevant retrieved instances within the top `k`. In this setting, the values are low: `3%, 8%, 13%, 20%` of queries get a relevant document within the top `k = 1, 5, 10, 50`, respectively. The modest increase across `k` suggests widening the candidate pool helps only slightly, which points to limited lexical overlap between query wording and document wording.
* Hit@k: whether at least one relevant document appears in the top `k`. These scores are much higher than recall, which indicates that BM25 retrieves something relevant, but usually not enough relevant documents to produce a strong recall. This is a useful sign: the retriever is not failing completely, but is not consistently ranking the best evidence very highly either.
* The gap between Recall@k and Hit@k also suggests qrels likely contain multiple relevant documents per query.
* MRR: this measures how early the first relevant document appears in the ranking. The value is moderate (0.4941), which suggests that when BM25 succeeds, it often places a relevant passage fairly high (on average rank 1/0.4941).
* The failure examples help explain this behavior: the retrieved documents have scores that are fairly close together, which suggests BM25 is finding documents with shared vocabulary, but not the correct evidence--natural, non-technical queries vs. medical, very technical language.
* BM25: a sparse lexical retriever. It ranks documents primarily by exact token overlap and term frequency signals, so it is strong when query and document wording align closely. Its main weakness are semantic mismatch, paraphrase sensitivity, and vocabulary mismatch. In this dataset, those weaknesses are visible because many queries are short and phrased differently from the supporting passages.

Overall, BM25 has limited coverage of the full relevant set, but when it succeeds, the first relevant passage is often ranked reasonably high. This makes it a useful lexical baseline, but not a strong retriever for this corpus. Together with the dataset characteristics, these results motivate further exploration of dense (semantic) embeddings, reranking, and hybrid retrieval pipelines.

#### 2. Dense Passage Retriever: Semantic Search with `SentenceTransformer` (BERT backbone) Embedding Model and Vector Database

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

In [8]:
class DenseRetriever(BaseRetriever):
    def __init__(self, corpus: Dict[str, str], model_name: str = "multi-qa-mpnet-base-cos-v1"):
        self.corpus = corpus
        self.doc_ids = list(corpus.keys())
        self.doc_texts = [corpus[doc_id] for doc_id in self.doc_ids]

        self.model = SentenceTransformer(model_name)

        doc_embeddings = self.model.encode(
            self.doc_texts,
            show_progress_bar=True
        ).astype("float32")

        faiss.normalize_L2(doc_embeddings)
        self.dim = doc_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(doc_embeddings)

        self.idx_to_docid = {idx: doc_id for idx, doc_id in enumerate(self.doc_ids)}


    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        query_emb = self.model.encode([query]).astype("float32")
        faiss.normalize_L2(query_emb)
        scores, indices = self.index.search(query_emb, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            doc_id = self.idx_to_docid[idx]
            score = float(score)
            results.append((doc_id, score))

        return results


    def get_name(self) -> str:
        return "Dense"

In [ ]:
dense = DenseRetriever(_corpus)

In [42]:
dense_scores = evaluate_retriever(dense, _queries, _relevant_docs)
for metric, value in dense_scores.items():
    print(f"{metric}: {value:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/114 [00:00<?, ?it/s]

recall@1: 0.0388
recall@5: 0.1052
recall@10: 0.1475
recall@50: 0.2387
hit@1: 0.4290
hit@5: 0.6296
hit@10: 0.6883
hit@50: 0.8086
mrr: 0.5192


* Dense retrieval improved over BM25 across all retrieval metrics, suggesting better semantic matching. However, absolute recall remains low, indicating that retrieval coverage is still limited and ranking remains a problem.
* SentenceTransformers uses a bi-encoder architecture using a BERT backbone. Queries and documents are encoded independently into fixed-length dense vectors. Document embeddings are pre-computed and stored in a FAISS index (vector database) for fast approximate nearest neighbor search.

#### 3. Hybrid Retrieval via Reciprocal Rank Fusion (RRF)

In [9]:
class HybridRetriever(BaseRetriever):
    def __init__(self, retrievers: List[BaseRetriever], rrf_k: int=60):
        self.retrievers = retrievers
        self.rrf_k = rrf_k


    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        all_results = {} # doc_id -> fusion_score

        for retriever in self.retrievers:
            results = retriever.search(query, k=k*2)

            for rank, (doc_id, _) in enumerate(results, start=1):
                rrf_score = 1.0 / (self.rrf_k + rank)
                all_results[doc_id] = all_results.get(doc_id, 0) + rrf_score

        sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:k]
        return [(doc_id, score) for doc_id, score in sorted_results]


    def get_name(self) -> str:
        names = [r.get_name() for r in self.retrievers]
        return f"Hybrid({'+'.join(names)})"

In [22]:
hybrid_bm25_dense = HybridRetriever([bm25, dense])
hybrid_scores = evaluate_retriever(hybrid_bm25_dense, _queries, _relevant_docs)
for metric, value in hybrid_scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0353
recall@5: 0.1114
recall@10: 0.1478
recall@50: 0.2493
hit@1: 0.4352
hit@5: 0.6235
hit@10: 0.7006
hit@50: 0.8148
mrr: 0.5278


* Although BM25 and dense retrieval capture complementary signals--BM25 handles exact lexical overlap and rare terms, while dense retrieval captures semantic similarity and paraphrase--all metrics are roughly in the same ballpark. Combining two retrievers that are already quite similar in performance evidently adds noise more than signal.
* The current issue is not just ranking quality but candidate recall. Hybrid retrieval and reranking can improve the ordering of retrieved documents, but it cannot recover documents that never enter the candidate set. Therefore, the next focus should be on improving retrieval coverage through chunking, query normalization (minor improvement), query expansion (w/ generative model: risky, costly), or a stronger retriever.

#### 4. Zero-Hit Analysis

In [24]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import random

In [19]:
@dataclass
class RetrieverStats:
    zero_hits:int = 0
    total_queries: int = 0

    @property
    def zero_hit_rate(self) -> float:
        return self.zero_hits / self.total_queries if self.total_queries > 0 else 0.0

@dataclass
class ZeroHitAnalysis:
    zero_hit_queries: List[Dict]
    count: int
    total_queries: int
    zero_hit_rate: float
    per_retriever_stats: Dict[str, RetrieverStats]

In [20]:
def analyze_zero_hits(
        queries: Dict[str, str],
        relevant_docs: Dict[str, Set[str]],
        retrievers: List[BaseRetriever],
        top_k: int = 100
) -> ZeroHitAnalysis:

    zero_hit_queries = []
    per_retriever_stats: Dict[str, RetrieverStats] = defaultdict(RetrieverStats)

    for qid, qtext in queries.items():
        if qid not in relevant_docs:
            continue

        rel = relevant_docs[qid]
        all_candidate_ids = set()

        for retriever in retrievers:
            candidates = retriever.search(qtext, k=top_k)
            candidate_ids = {doc_id for doc_id, _ in candidates}
            all_candidate_ids.update(candidate_ids)

            ret_name = retriever.get_name()
            per_retriever_stats[ret_name].total_queries += 1
            if len(rel.intersection(all_candidate_ids)) == 0:
                per_retriever_stats[ret_name].zero_hits += 1

        if len(rel.intersection(all_candidate_ids)) == 0:
            zero_hit_queries.append({
                "qid": qid,
                "query": qtext,
                "relevant_docs": rel,
                "num_relevant": len(rel)
            })

    total_queries_with_rel = len([q for q in queries if q in relevant_docs])

    return ZeroHitAnalysis(
        zero_hit_queries=zero_hit_queries,
        count=len(zero_hit_queries),
        total_queries=total_queries_with_rel,
        zero_hit_rate=len(zero_hit_queries) / total_queries_with_rel if total_queries_with_rel > 0 else 0.0,
        per_retriever_stats=per_retriever_stats
    )

In [21]:
def print_zero_hit_summary(analysis: ZeroHitAnalysis):
    print(f"total queries with relevance judgments: {analysis.total_queries}")
    print(f"queries with zero hits (k={TOP_K}): {analysis.count}")
    print(f"zero-hit rate: {analysis.zero_hit_rate:.2%}")
    print("\nper-retriever zero-hit rates:")
    for ret_name, stats in analysis.per_retriever_stats.items():
        print(f"  {ret_name:10s}:{stats.zero_hits:3d}/{stats.total_queries:3d} "
              f"({stats.zero_hit_rate:.2%})")

In [22]:
def snippet(text:str, length: int = 160) -> str:
    if not text:
        return "[EMPTY]"
    text = text.replace("\n", " ").replace("\r", " ").strip()
    text = ' '.join(text.split())
    return text[:length] + ("..." if len(text) > length else "")

def inspec_zero_hit_query(
        qid: str,
        query: str,
        relevant_docs: Set[str],
        corpus: Dict[str, str],
        retrievers: List[BaseRetriever],
        top_k: int = 10,
        show_snippets: bool = True
):
    """inspection of a single zero-hit query"""
    print("\n" + "=" * 100)
    print(f"QID       : {qid}")
    print(f"QUERY     : {query}")
    print(f"REL ({len(relevant_docs)}): {', '.join(sorted(relevant_docs))}")
    for retriever in retrievers:

        print(f"\n{retriever.get_name().upper()}:")
        results = retriever.search(query, k=top_k)

        for i, (doc_id, score) in enumerate(results, start=1):
            is_relevant = doc_id in relevant_docs
            marker = "?" if is_relevant else "  "
            print(f"  {marker} {i:>2}. {doc_id} score={score:.4f}")
            if show_snippets and doc_id in corpus:
                print(f"        {snippet(corpus[doc_id])}")

def analyze_term_overlap(
        query: str,
        corpus: Dict[str, str],
        retrievers: List[BaseRetriever],
        top_k: int = 10,
):
    query_terms = set(query.lower().split())
    print(f"\nQuery terms: {query_terms}")

    for retriever in retrievers:
        results = retriever.search(query, k=top_k)
        retrieved_texts = [corpus.get(doc_id, "") for doc_id, _ in results if doc_id in corpus]

        all_terms = []
        for text in retrieved_texts:
            all_terms.extend(text.split())

        term_freq = Counter(all_terms)

        print(f"\n{retriever.get_name()} - Top terms in retrieved docs:")
        for term, freq in term_freq.most_common(10):
            if term not in query_terms:
                print(f"    {term}: {freq}")


In [37]:
TOP_K = 100
retrievers = [bm25, dense, hybrid_bm25_dense]
analysis = analyze_zero_hits(_queries, _relevant_docs, retrievers, top_k=TOP_K)
print_zero_hit_summary(analysis)

zero_hits = analysis.zero_hit_queries
if zero_hits:
    sample_size = min(5, len(zero_hits))
    sample = random.sample(zero_hits, sample_size)

    for item in sample:
        inspec_zero_hit_query(
            qid=item["qid"],
            query=item["query"],
            relevant_docs=item["relevant_docs"],
            corpus=_corpus,
            retrievers=retrievers,
            top_k=10,
            show_snippets=True
        )

        analyze_term_overlap(
            query=item["query"],
            corpus=_corpus,
            retrievers=retrievers,
            top_k=10,
        )

total queries with relevance judgments: 324
queries with zero hits (k=100): 39
zero-hit rate: 12.04%

per-retriever zero-hit rates:
  BM25      : 77/324 (23.77%)
  Dense     : 39/324 (12.04%)
  Hybrid(BM25+Dense): 39/324 (12.04%)

QID   : PLAIN-3096
QUERY : Phytochemicals: The Nutrition Facts Missing From the Label
REL (8): MED-2215, MED-4390, MED-4585, MED-4587, MED-4588, MED-4589, MED-4590, MED-726

BM25:
      1. MED-2720 score=21.9397
        Potential effect of physical activity based menu labels on the calorie content of selected fast food meals. In this study we examined the effect of physical act...
      2. MED-4686 score=21.3421
        Proposal for a dietary "phytochemical index". There is ample reason to believe that diets rich in phytochemicals provide protection from vascular diseases and m...
      3. MED-2493 score=21.1098
        Environmental toxicants and the developing immune system: a missing link in the global battle against infectious disease? There is now compel

* Most failures were caused by lexical mismatch, short ambiguous queries, and multi-concept queries. A smaller subset appears to be due to document granularity and domain-specific vocabulary differences.

#### 5. Query Expansion with Sparse Lexical and Expansion (SPLADE) models

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from scipy.sparse import csr_array

In [11]:
class SpladeRetriever(BaseRetriever):
    def __init__(self, corpus: Dict[str, str], model_name: str = "naver/splade-cocondenser-ensembledistil"):
        self.corpus = corpus
        self.doc_ids = list(corpus.keys())
        self.doc_texts = [corpus[doc_id] for doc_id in self.doc_ids]

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(self.device)
        self.model.eval()

        self.vocab_size = self.model.config.vocab_size

        self.doc_vectors = []
        batch_size = 32

        for i in range(0, len(self.doc_texts), batch_size):
            batch = self.doc_texts[i:i + batch_size]
            batch_vectors = self._encode_batch(batch, is_query=False)
            self.doc_vectors.extend(batch_vectors)

            if i % 100 == 0:
                print(f"  encoded {i}/{len(self.doc_ids)}")


    def _encode_batch(self, texts: List[str], is_query: bool = False) -> List:
        """encode a batch of texts to sparse vectors"""
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512,
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            attention_mask = inputs.attention_mask.unsqueeze(-1)

            relu_log = torch.log(1 + torch.relu(logits))
            weighted_log = relu_log * attention_mask
            vectors = torch.max(weighted_log, dim=1)[0] # (batch, vocab size)

        sparse_vectors = []
        for vec in vectors:
            mask = vec > 1e-5 # boolean mask where values > 0.00001 to reduce memory usage
            idx = torch.nonzero(mask).squeeze().cpu().numpy() # get indices where mask is True
            values = vec[mask].cpu().numpy()                  # get values at those indices

            if idx.size == 0:
                idx = np.array([0])
                values = np.array([0.0])
            elif idx.ndim == 0:
                idx = np.array([idx])
                values = np.array([values])

            sparse_vectors.append(
                csr_array((values, (np.zeros(len(idx)), idx)), shape=(1, self.vocab_size))
            )

        return sparse_vectors


    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        query_vec = self._encode_batch([query], is_query=True)[0]

        scores = []
        for doc_vec in self.doc_vectors:
            sim = (query_vec @ doc_vec.T)[0, 0]
            scores.append(sim)

        scores = np.array(scores)
        top_indices = np.argsort(scores)[::-1][:k]

        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append((self.doc_ids[idx], scores[idx]))

        return results

    def get_name(self) -> str:
        return "SPLADE"

In [12]:
splade = SpladeRetriever(_corpus)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

  encoded 0/3633
  encoded 800/3633
  encoded 1600/3633
  encoded 2400/3633
  encoded 3200/3633


In [38]:
splade_scores = evaluate_retriever(splade, _queries, _relevant_docs)
for metric, value in splade_scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0351
recall@5: 0.1204
recall@10: 0.1581
recall@50: 0.2460
hit@1: 0.4537
hit@5: 0.6790
hit@10: 0.7346
hit@50: 0.7963
mrr: 0.5501


In [44]:
hybrid_splade_dense = HybridRetriever([dense, splade])
hybrid_sd_scores = evaluate_retriever(hybrid_splade_dense, _queries, _relevant_docs)
for metric, value in hybrid_sd_scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0373
recall@5: 0.1138
recall@10: 0.1549
recall@50: 0.2679
hit@1: 0.4228
hit@5: 0.6451
hit@10: 0.7006
hit@50: 0.8395
mrr: 0.5294


**Notes**
* SPLADE performs learned sparse retrieval using a BERT backbone using three key operations:
    * ReLU activation: zeroes out negative values
    * Log transformation: `log(1 + ReLU(x))` compresses range
    * Max pooling: keeps highest score per vocabulary term
* The output is a sparse representation via `csr_array`, where only non-zero values + their positions are stored. This enables mapping these queries and documents to a high-dimensional sparse space, effectively performing query and document expansion. Unfortunately, SPLADE does not fully solve the issue of ambiguous queries, but it shows signs of better generalizability.
* Best overall metrics. Combines BM25's explicit term matching with neural semantic expansion. Only downside: slower inference and dense beats it at Recall@50.

#### 6. Fine-Tuning Dense Retriever with Contrastive Learning on NFcorpus

In [13]:
from sentence_transformers import SentenceTransformer

from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments

from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss, TripletLoss, TripletDistanceMetric
from sentence_transformers.base.sampler import BatchSamplers
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import mine_hard_negatives

from peft import LoraConfig, TaskType, get_peft_model

import torch
from datasets import Dataset
from collections import defaultdict
import ir_datasets

In [14]:
class DenseRetrieverTrainer:
    def __init__(
        self,
        base_model_name: str = "multi-qa-mpnet-base-cos-v1",
        max_seq_length: int = 256,
    ):
        self.base_model_name = base_model_name
        self.max_seq_length = max_seq_length
        self.model = None

        self._lora_config = None
        self._mining_config = None
        self._mnrl_config = None
        self._triplet_config = None
        self._eval_config = None
        self._training_config = {}


    def with_lora(self, r: int = 16, alpha: int = 32, dropout: float = 0.1,
                  target_modules: list = None) -> "DenseRetrieverTrainer":
        self._lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=r,
            lora_alpha=alpha,
            lora_dropout=dropout,
            target_modules=target_modules or ['query', 'key', 'value', 'dense'],
        )
        return self


    def with_hard_negatives(self, num_negatives: int = 5, range_min: int = 10,
                            range_max: int = 50, max_score: float = 0.8,
                            relative_margin: float = 0.05,
                            sampling_strategy: str = "top",
                            batch_size: int = 128) -> "DenseRetrieverTrainer":
        self._mining_config = {
            'num_negatives': num_negatives,
            'range_min': range_min,
            'range_max': range_max,
            'max_score': max_score,
            'relative_margin': relative_margin,
            'sampling_strategy': sampling_strategy,
            'batch_size': batch_size,
            'use_faiss': True
        }
        return self


    def with_mnrl_config(self, hardness_mode: str = "in_batch_ngeatives", hardness_strength: float = 9.0) -> "DenseRetrieverTrainer":
        self._mnrl_config = {
            'hardness_mode': hardness_mode,
            'hardness_strength': hardness_strength
        }
        return self


    def with_triplet_loss(self, margin: float = 0.5) -> "DenseRetrieverTrainer":
        self._triplet_config = {'margin': margin}
        return self


    def with_evaluation(self, dataset, name: str = "nfcorpus-dev",
                    batch_size: int = 64) -> "DenseRetrieverTrainer":
        self._eval_config = {
            'dataset': dataset,
            'name': name,
            'batch_size': batch_size
        }
        return self


    def with_training_args(self, **kwargs) -> "DenseRetrieverTrainer":
        self._training_config.update(kwargs)
        return self


    # noinspection PyMethodMayBeStatic
    def prepare_pairs(self, dataset, relevance_threshold: int = 1):
        """convert ir_datasets to HuggingFace Dataset format"""
        pairs = [] # (query, positive_doc)

        # lookup dicts
        query_dict = {q.query_id: q.text for q in dataset.queries_iter()}
        doc_dict = {
            d.doc_id: f"{d.title} {d.text}".strip()
            for d in dataset.docs_iter()
        }

        for qrel in dataset.qrels_iter():
            if qrel.relevance >= relevance_threshold:
                query_text = query_dict.get(qrel.query_id)
                doc_text = doc_dict.get(qrel.doc_id)
                if query_text and doc_text:
                    pairs.append((query_text, doc_text))

        return Dataset.from_dict({
            "anchor": [p[0] for p in pairs],
            "positive": [p[1] for p in pairs],
        })


    # noinspection PyMethodMayBeStatic
    def prepare_triplets(self, dataset, num_negatives: int = 1):
        """converts dataset to triplet format. Expects dataset from mine_hard_negatives()"""
        triplets = []

        for example in dataset:
            anchor = example["anchor"]
            positive = example["positive"]
            negative = example.get(f"negative_{num_negatives}", example.get("negative_1"))

            if negative:
                triplets.append((anchor, positive, negative))

        return Dataset.from_dict({
            "anchor": [t[0] for t in triplets],
            "positive": [t[1] for t in triplets],
            "negative": [t[2] for t in triplets],
        })


    # noinspection PyMethodMayBeStatic
    def prepare_ir_evaluator(self, dataset, name: str = "nfcorpus-dev", batch_size: int = 64):
        queries = {q.query_id: q.text for q in dataset.queries_iter()}
        corpus = {
            d.doc_id: f"{d.title} {d.text}".strip()
            for d in dataset.docs_iter()
        }
        relevant_docs = defaultdict(set)
        for qrel in dataset.qrels_iter():
            if qrel.relevance >= 1:
                relevant_docs[qrel.query_id].add(qrel.doc_id)

        return InformationRetrievalEvaluator(
            corpus=corpus,
            queries=queries,
            relevant_docs=dict(relevant_docs),
            name=name,
            batch_size=batch_size,
            show_progress_bar=False,
            write_csv=False,
        )


    def mine_hard_negatives(self, dataset):
        if self.model is None:
            self.model = SentenceTransformer(self.base_model_name)

        if self._mining_config is None:
            raise ValueError("call with_hard_negatives() before mine_hard_negatives()")

        mined_dataset = mine_hard_negatives(
            dataset=dataset,
            model=self.model,
            range_min=self._mining_config['range_min'],
            range_max=self._mining_config['range_max'],
            max_score=self._mining_config['max_score'],
            relative_margin=self._mining_config['relative_margin'],
            num_negatives=self._mining_config['num_negatives'],
            sampling_strategy=self._mining_config['sampling_strategy'],
            batch_size=self._mining_config['batch_size'],
            use_faiss=self._mining_config['use_faiss'],
        )

        print(f"Mined {len(mined_dataset)} examples with {self._mining_config['num_negatives']} negatives each")
        return mined_dataset


    def _create_loss(self):
        if self._triplet_config is not None:
            return TripletLoss(
                self.model,
                distance_metric=TripletDistanceMetric.COSINE,
                triplet_margin=self._triplet_config['margin']
            )

        if self._mnrl_config:
            return MultipleNegativesRankingLoss(
                self.model,
                hardness_mode=self._mnrl_config['hardness_mode'],
                hardness_strength=self._mnrl_config['hardness_strength']
            )

        return MultipleNegativesRankingLoss(self.model)


    def train(self, output_path: str, train_dataset, eval_dataset=None):
        self.model = SentenceTransformer(self.base_model_name)
        self.model.max_seq_length = self._training_config.get('max_seq_length', self.max_seq_length)

        if self._lora_config is not None:
            self.model[0].auto_model = get_peft_model(
                self.model[0].auto_model,
                self._lora_config
            )
            trainable = sum(p.numel() for p in self.model[0].parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.model.parameters())
            print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

        if self._mining_config is not None:
            train_dataset = self.mine_hard_negatives(train_dataset)

        loss = self._create_loss()

        evaluator = None
        if self._eval_config is not None:
            evaluator = self.prepare_ir_evaluator(
                dataset=self._eval_config['dataset'],
                name=self._eval_config.get('name', 'nfcorpus-dev'),
                batch_size=self._eval_config.get('batch_size', 64),
            )
        elif eval_dataset is not None:
            evaluator = self.prepare_ir_evaluator(
                dataset=eval_dataset,
                name='declared-eval-dataset',
                batch_size=64,
            )

        args = SentenceTransformerTrainingArguments(
            output_dir=output_path,
            fp16=torch.cuda.is_available(),
            num_train_epochs=self._training_config.get('num_epochs', 3),
            per_device_train_batch_size=self._training_config.get('batch_size', 16),
            gradient_accumulation_steps=self._training_config.get('gradient_accumulation_steps', 1),
            learning_rate=self._training_config.get('learning_rate', 2e-5),
            logging_strategy=self._training_config.get('logging_strategy', 'epoch'),
            save_strategy=self._training_config.get('save_strategy', 'epoch'),
            save_total_limit=self._training_config.get('save_total_limit', 2),
            eval_strategy=self._training_config.get('eval_strategy', 'epoch') if evaluator else 'no',
            load_best_model_at_end=self._training_config.get('load_best_model_at_end', True) if evaluator else False,
            metric_for_best_model=self._training_config.get(
                "metric_for_best_model",
                f"eval_{self._eval_config.get('name', 'nfcorpus-dev')}_cosine_mrr@10" if self._eval_config else None
            ),
            greater_is_better=True,
            report_to="none",
            batch_sampler=BatchSamplers.NO_DUPLICATES,
            run_name=self._training_config.get('run_name', output_path.split('/')[-1]),
        )

        trainer = SentenceTransformerTrainer(
            model=self.model,
            args=args,
            train_dataset=train_dataset,
            loss=loss,
            evaluator=evaluator,
        )

        trainer.train()
        self.model.save_pretrained(output_path)

        final_metrics = None
        if evaluator is not None:
            final_metrics = evaluator(self.model)
            print(f"Final retrieval metric ({evaluator.primary_metric}): "
                  f"{final_metrics[evaluator.primary_metric]:.4f}")

        return self.model, final_metrics

In [3]:
# data preparation
train_raw = ir_datasets.load("beir/nfcorpus/train")
dev_raw = ir_datasets.load("beir/nfcorpus/dev")

_trainer = DenseRetrieverTrainer()
train_pairs = _trainer.prepare_pairs(train_raw, relevance_threshold=1)

In [6]:
len(train_pairs)

110575

In [13]:
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

#### 6.1 in-batch negatives with Multiple Negatives Ranking Loss

In [6]:
trainer_baseline = (DenseRetrieverTrainer()
    .with_evaluation(dev_raw, name="nfcorpus-dev"))

model_baseline, metrics_baseline = trainer_baseline.train(
    train_dataset=train_pairs,
    output_path="D:/models/nfcorpus-dense-ft-baseline",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': '1.953', 'grad_norm': '12.33', 'learning_rate': '1.334e-05', 'epoch': '1'}
{'eval_nfcorpus-dev_cosine_accuracy@1': '0.4352', 'eval_nfcorpus-dev_cosine_accuracy@3': '0.5833', 'eval_nfcorpus-dev_cosine_accuracy@5': '0.6327', 'eval_nfcorpus-dev_cosine_accuracy@10': '0.6944', 'eval_nfcorpus-dev_cosine_precision@1': '0.4352', 'eval_nfcorpus-dev_cosine_precision@3': '0.3776', 'eval_nfcorpus-dev_cosine_precision@5': '0.3593', 'eval_nfcorpus-dev_cosine_precision@10': '0.2966', 'eval_nfcorpus-dev_cosine_recall@1': '0.03766', 'eval_nfcorpus-dev_cosine_recall@3': '0.08162', 'eval_nfcorpus-dev_cosine_recall@5': '0.1246', 'eval_nfcorpus-dev_cosine_recall@10': '0.1768', 'eval_nfcorpus-dev_cosine_ndcg@10': '0.3619', 'eval_nfcorpus-dev_cosine_mrr@10': '0.5219', 'eval_nfcorpus-dev_cosine_map@100': '0.2181', 'eval_runtime': '30.21', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.484', 'grad_norm': '15.38', 'learning_rate': '6.678e-06', 'epoch': '2'}
{'eval_nfcorpus-dev_cosine_accuracy@1': '0.4568', 'eval_nfcorpus-dev_cosine_accuracy@3': '0.5648', 'eval_nfcorpus-dev_cosine_accuracy@5': '0.6142', 'eval_nfcorpus-dev_cosine_accuracy@10': '0.6914', 'eval_nfcorpus-dev_cosine_precision@1': '0.4568', 'eval_nfcorpus-dev_cosine_precision@3': '0.3858', 'eval_nfcorpus-dev_cosine_precision@5': '0.3531', 'eval_nfcorpus-dev_cosine_precision@10': '0.3099', 'eval_nfcorpus-dev_cosine_recall@1': '0.04211', 'eval_nfcorpus-dev_cosine_recall@3': '0.08525', 'eval_nfcorpus-dev_cosine_recall@5': '0.1217', 'eval_nfcorpus-dev_cosine_recall@10': '0.1825', 'eval_nfcorpus-dev_cosine_ndcg@10': '0.3728', 'eval_nfcorpus-dev_cosine_mrr@10': '0.526', 'eval_nfcorpus-dev_cosine_map@100': '0.2347', 'eval_runtime': '30.12', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '2'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1.017e+04', 'train_samples_per_second': '32.6', 'train_steps_per_second': '2.038', 'train_loss': '1.572', 'epoch': '3'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final retrieval metric (nfcorpus-dev_cosine_ndcg@10): 0.3728


**NOTES**
* MultipleNegativesRankingLoss enforces achieving semantic search rather than lexical overlap. The goal is to create a clear distinction between relevant (positive) and irrelevant (negative) data points. This distinction is achieved by maximizing the similarity between query_i and doc_i, while minimizing the similarity between query_i and all negatives (doc_j, j≠i). The loss function applies softmax normalization (`exp(sim(q_i, d_i)) / Σ_j exp(sim(q_i, d_j)`) to convert similarities into probabilities, then computes the negative log likelihood of positive pairs.
* Overall diminishing returns. The base model (`multi-qa-mpnet-base-cos-v1`) was already fine-tuned on MS MACRO (web search). NFCorpus (5k pairs, medical QA) is too small and perhaps similar to meaningfully improve it. Larger batch size to increase in-batch negatives and alternative losses might yield better gains.

#### 6.1 Fine-Tuing with LoRA

In [ ]:
trainer_lora = (DenseRetrieverTrainer()
    .with_lora(r=16, alpha=32)
    .with_training_args(earning_rate=2e-4,)
    .with_evaluation(dev_raw, name="nfcorpus-dev")
)

model_lora, metrics_lora = trainer_lora.train(
    output_path="D:/models/nfcorpus-dense-lora",
    train_dataset=train_pairs,
)

* Fine-tuning with LoRA is significantly outperformed by both SPLADE (sparse) and full fine-tuned dense retrieval. This is likely because NFcorpus, being a small, specialized domain, requires full weight adaptation rather than low-rank updates. While LoRA offers memory efficiency, this advantage can be foregone in favor of achieving better retrieval performance.
| Metric | SPLADE | Full Fine-tuned | LoRA |
|--------|--------|-----------------|------|
| **MRR@10** | 0.5294 | 0.5202 | 0.4853 |
| **NDCG@10** | -- | 0.3792 | 0.3064 |
| **Recall@10** | 0.1549 | 0.1947 | 0.1418 |
| **Training Loss** | -- | 1.281 | 2.152 |

#### 6.2 Hardness-weighted MNRL

In [5]:
trainer_hard = (DenseRetrieverTrainer()
    .with_hard_negatives(
        num_negatives=2,
        range_min=10,
        range_max=100,
        relative_margin=0.01,
        max_score=0.85,
        sampling_strategy="top"
    )
    .with_evaluation(dev_raw, name="nfcorpus-dev")
)

model_hard, metrics_hard = trainer_hard.train(
    output_path="D:/models/nfcorpus-dense-hard-negatives",
    train_dataset=train_pairs,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Found 2576 unique queries out of 110575 total queries.
Found an average of 42.925 positives per query.


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  6.90it/s]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2580', 'train_samples_per_second': '0.923', 'train_steps_per_second': '0.058', 'train_loss': '0.2479', 'epoch': '3'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final retrieval metric (nfcorpus-dev_cosine_ndcg@10): 0.3232


**NOTES**
* Base configuration `{ num_negatives=3, range_min=5, range_max=10, max_score=0.8, relative_margin=0.05, sampling_strategy="random" }` yielded 694 mined examples. With many relevant docs per query (42.9 avg), the model likely finds that many candidates are too similar to the positive and filters them out. This suggest `relative_margin=0.05` is too aggressive for NFcorpus's dense positive space.
* Nearly maxed out configuration `{ num_negatives=3, range_min=5, range_max=200, max_score=0.9, relative_margin=0.005, sampling_strategy="random" }` yielded 1613 samples (10.8% of theoretical max). While this maximizes quanity, it raises concerns about false negatives and reinforces that true hard negatives are genuinely rare for this dataset.
* Minimal configuration: `{ num_negatives=1, range_min=10, range_max=100, max_score=0.8, relative_margin=0.02, sampling_strategy="top" }` yielded 374 examples -- too few for stable training.
* Given the constraints (43 positives/query, rare hard negatives), our baseline configuration prioritizes quality over quantity

In [7]:
trainer_base_weighted = (DenseRetrieverTrainer()
    .with_mnrl_config(hardness_mode="in_batch_negatives", hardness_strength=9.0)
    .with_training_args(batch_size=16, gradient_accumulation_steps=2)
    .with_evaluation(dev_raw, name="nfcorpus-dev")
)
model, metrics = trainer_base_weighted.train("D:/models/hardness-weighted", train_pairs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': '4.472', 'grad_norm': '19.1', 'learning_rate': '1.334e-05', 'epoch': '1'}
{'eval_nfcorpus-dev_cosine_accuracy@1': '0.4321', 'eval_nfcorpus-dev_cosine_accuracy@3': '0.5833', 'eval_nfcorpus-dev_cosine_accuracy@5': '0.6358', 'eval_nfcorpus-dev_cosine_accuracy@10': '0.7037', 'eval_nfcorpus-dev_cosine_precision@1': '0.4321', 'eval_nfcorpus-dev_cosine_precision@3': '0.3889', 'eval_nfcorpus-dev_cosine_precision@5': '0.3543', 'eval_nfcorpus-dev_cosine_precision@10': '0.2951', 'eval_nfcorpus-dev_cosine_recall@1': '0.03285', 'eval_nfcorpus-dev_cosine_recall@3': '0.08661', 'eval_nfcorpus-dev_cosine_recall@5': '0.1195', 'eval_nfcorpus-dev_cosine_recall@10': '0.1804', 'eval_nfcorpus-dev_cosine_ndcg@10': '0.3589', 'eval_nfcorpus-dev_cosine_mrr@10': '0.5225', 'eval_nfcorpus-dev_cosine_map@100': '0.215', 'eval_runtime': '47.14', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '2.933', 'grad_norm': '21.64', 'learning_rate': '6.677e-06', 'epoch': '2'}
{'eval_nfcorpus-dev_cosine_accuracy@1': '0.4444', 'eval_nfcorpus-dev_cosine_accuracy@3': '0.5772', 'eval_nfcorpus-dev_cosine_accuracy@5': '0.6173', 'eval_nfcorpus-dev_cosine_accuracy@10': '0.6667', 'eval_nfcorpus-dev_cosine_precision@1': '0.4444', 'eval_nfcorpus-dev_cosine_precision@3': '0.4074', 'eval_nfcorpus-dev_cosine_precision@5': '0.3796', 'eval_nfcorpus-dev_cosine_precision@10': '0.3253', 'eval_nfcorpus-dev_cosine_recall@1': '0.03509', 'eval_nfcorpus-dev_cosine_recall@3': '0.08503', 'eval_nfcorpus-dev_cosine_recall@5': '0.1274', 'eval_nfcorpus-dev_cosine_recall@10': '0.1888', 'eval_nfcorpus-dev_cosine_ndcg@10': '0.3862', 'eval_nfcorpus-dev_cosine_mrr@10': '0.5207', 'eval_nfcorpus-dev_cosine_map@100': '0.2358', 'eval_runtime': '30.97', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '2'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1.103e+04', 'train_samples_per_second': '30.08', 'train_steps_per_second': '1.88', 'train_loss': '3.272', 'epoch': '3'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final retrieval metric (nfcorpus-dev_cosine_ndcg@10): 0.3913


In [5]:
trainer_hard_weighted = (DenseRetrieverTrainer()
    .with_hard_negatives(
        num_negatives=2,
        range_min=10,
        range_max=100,
        relative_margin=0.01,
        max_score=0.85,
        sampling_strategy="top")
    .with_mnrl_config(hardness_mode="hard_negatives", hardness_strength=9.0)
    .with_training_args(batch_size=16, gradient_accumulation_steps=2)
    .with_evaluation(dev_raw, name="nfcorpus-dev")
)
model, metrics = trainer_hard_weighted.train("D:/models/nfcorpus-dense-mnrl-hard", train_pairs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Found 2576 unique queries out of 110575 total queries.
Found an average of 42.925 positives per query.


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  5.94it/s]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2753', 'train_samples_per_second': '0.865', 'train_steps_per_second': '0.027', 'train_loss': '0.449', 'epoch': '3'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final retrieval metric (nfcorpus-dev_cosine_ndcg@10): 0.3262


**NOTES**
* Explicit hard negative mining proves detrimental due to the dataset's dense positive space, which increases false negative risk and decreases generalizability. In-batch negatives from MNR provided sufficient contrastive signal without the nosie of mined hard negatives, albeit with significantly longer training time.
* Overall fine-tuning with contrastive learning--particularly with just in-batch negatives for this dataset--yielded roughly a 4% increase in recall (0.1549 → 0.1947). MRR remained competitive with SPLADE (0.5202 vs. 0.5294), suggesting dense retrieval casts a wider net while SPLADE excels at ranking precision.
* The new hardness weighting feature (sentence-transformers v5.3.0) with increased effective batch sized pushed MRR slightly above SPLADE (0.5327) while maintaining Recall gains. However, the improvement is marginal and may not justify the significant training time increase.
| Metric            | SPLADE | MNRL (in-batch) | + Hardness Weighting* | Hard Negatives | + Hardness Weighting* |
|-------------------|--------|----------------|-----------------------|----------------|-----------------------|
| **MRR@10**        | 0.5294 | 0.5202         | 0.5327                | 0.5022              | 0.4991                     |
| **NDCG@10**       | TODO      | 0.3792         | 0.3913                | 0.3245              | 0.3262                     |
| **Recall@10**     | 0.1549 | 0.1947         | 0.1963                | 0.1544              | 0.1551                     |
| **Training Loss** | —      | 1.281          | 2.412                 | 0.1365              | 0.449                     |

> *also increased effective batch size for more potential negatives.

#### 6.3 Hybrid SPLADE + Fine-tuned Dense

###### initialize section 0, 2, and 5

In [12]:
# section 0, 5
splade = SpladeRetriever(_corpus)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

  encoded 0/3633
  encoded 800/3633
  encoded 1600/3633
  encoded 2400/3633
  encoded 3200/3633


In [15]:
# section 0, 2
dense = DenseRetriever(_corpus, model_name="D:/models/hardness-weighted")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/114 [00:00<?, ?it/s]

In [16]:
# section 3
hybrid_splade_dense = HybridRetriever([splade, dense])

In [16]:
hybrid_scores = evaluate_retriever(hybrid_splade_dense, _queries, _relevant_docs)
for metric, value in hybrid_scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0389
recall@5: 0.1294
recall@10: 0.1833
recall@50: 0.3504
hit@1: 0.4568
hit@5: 0.6605
hit@10: 0.7284
hit@50: 0.8580
mrr: 0.5557


In [25]:
# section 4
TOP_K = 100
retrievers = [splade, dense, hybrid_splade_dense]
analysis = analyze_zero_hits(_queries, _relevant_docs, retrievers, top_k=TOP_K)
print_zero_hit_summary(analysis)

zero_hits = analysis.zero_hit_queries
if zero_hits:
    sample_size = min(5, len(zero_hits))
    sample = random.sample(zero_hits, sample_size)

    for item in sample:
        inspec_zero_hit_query(
            qid=item["qid"],
            query=item["query"],
            relevant_docs=item["relevant_docs"],
            corpus=_corpus,
            retrievers=retrievers,
            top_k=10,
            show_snippets=True
        )

        analyze_term_overlap(
            query=item["query"],
            corpus=_corpus,
            retrievers=retrievers,
            top_k=10,
        )

total queries with relevance judgments: 324
queries with zero hits (k=100): 29
zero-hit rate: 8.95%

per-retriever zero-hit rates:
  SPLADE    : 50/324 (15.43%)
  Dense     : 29/324 (8.95%)
  Hybrid(SPLADE+Dense): 29/324 (8.95%)

QID       : PLAIN-871
QUERY     : champagne grapes
REL (1): MED-5081

SPLADE:
      1. MED-3207 score=12.6754
        The grapefruit: an old wine in a new glass? Metabolic and cardiovascular perspectives Summary Grapefruit is a popular, tasty and nutritive fruit enjoyed globall...
      2. MED-398 score=12.6754
        The grapefruit: an old wine in a new glass? Metabolic and cardiovascular perspectives Summary Grapefruit is a popular, tasty and nutritive fruit enjoyed globall...
      3. MED-2083 score=11.9068
        Grape juice, but not orange juice or grapefruit juice, inhibits human platelet aggregation. Coronary artery disease is responsible for much mortality and morbid...
      4. MED-2669 score=11.1031
        Concord grape juice supplementation impro

* Hybrid retrieval nearly doubled Recall@50 (0.201 → 0.3504) and improved Hit@50 (0.7407 → 0.8580) compared to the BM25 baseline. Recall@10 is slightly below Dense alone but moderately improved over SPLADE, preserving some benefits of keyword-base retrieval. This slight dip is offset by a meaningful MRR improvement (0.5327 → 0.5557).
* Hybridization eliminates SPLADE's zero-hit queries (50 → 29), matching Dense alone's coverage while adding SPLADE's ranking precision. The zero-hit rate now stands at 8.95% (29/324 queries).
* **Remaining Challenges**: zero-hits persist for queries with few relevant documents or ambiguous terminology (e.g., drug names like "acarbose"). These cases would benefit from a larger document store or external knowledge augmentation.
    * The retrieved document terms are dominated by stopwords (prepositions, articles), though meaning is likely still captured via similarity scores. Pre-processing could clean up term frequency analysis but is not necessary for retrieval quality.

#### 7. Reranking

###### 7.1 Cross-Encoder

In [17]:
from sentence_transformers import CrossEncoder
from typing import List, Tuple, Dict
import numpy as np

In [18]:
class Reranker:
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.model = CrossEncoder(model_name)
        self.model_name = model_name

    def rerank(
        self,
        query: str,
        candidates: List[Tuple[str, str]],
        top_k: int = 10
    ) -> List[Tuple[str, float]]:
        pairs = [(query, doc_text) for _, doc_text in candidates]
        scores = self.model.predict(pairs)
        normalized_scores = 1 / (1 + np.exp(-scores))

        results = [
            (candidates[i][0], float(normalized_scores[i]))
            for i in range (len(candidates))
        ]
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]


    def rerank_from_retriever(
        self,
        query: str,
        retriever,
        retrieve_k: int = 50,
        rerank_k: int = 10
    ) -> List[Tuple[str, float]]:
        candidates = retriever.search(query, k=retrieve_k)
        candidate_texts = [
            (doc_id, retriever.corpus.get(doc_id, ""))
            for doc_id, _ in candidates
        ]
        return self.rerank(query, candidate_texts, top_k=rerank_k)

In [30]:
class RerankRetriever(BaseRetriever):
    def __init__(self, retriever: BaseRetriever, corpus: Dict[str, str],
                 rerank_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
                 retrieve_k: int = 50, rerank_weight: float = 0.6,
                 blend_strategy: str = ""):
        self.retriever = retriever
        self.corpus = corpus
        self.reranker = Reranker(rerank_model)
        self.retrieve_k = retrieve_k
        self.rerank_weight = rerank_weight
        self.blend_strategy = blend_strategy


    def _normalize(self, scores: List[float]) -> List[float]:
        min_score = min(scores)
        max_score = max(scores)
        if max_score > min_score:
            return [(s - min_score) / (max_score - min_score) for s in scores]
        return [0.5] * len(scores)


    def search(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        candidates = self.retriever.search(query, k=self.retrieve_k)

        if not candidates:
            return[]

        candidate_texts = [(doc_id, self.corpus.get(doc_id, ""))
                          for doc_id, _ in candidates]

        rerank_results = self.reranker.rerank(query, candidate_texts, top_k=len(candidates))
        rerank_dict = {doc_id: score for doc_id, score in rerank_results}
        rerank_scores = [rerank_dict.get(doc_id, 0.0) for doc_id, _ in candidates]

        first_stage_scores = [score for _, score in candidates]
        norm_first_stage = self._normalize(first_stage_scores) # compatability with sigmoid

        blended = []
        for i, (doc_id, _) in enumerate(candidates):
            if self.blend_strategy == "weighted":
                final = (1 - self.rerank_weight) * norm_first_stage[i] + self.rerank_weight * rerank_scores[i]
            elif self.blend_strategy == "position_aware":
                if i < 3:
                    final = 0.75 * rerank_scores[i] + 0.25 * norm_first_stage[i]
                elif i < 10:
                    final = 0.60 * rerank_scores[i] + 0.40 * norm_first_stage[i]
                else:
                    final = 0.40 * rerank_scores[i] + 0.60 * norm_first_stage[i]
            else:
                final = rerank_scores[i]

            blended.append((doc_id, final))

        blended.sort(key=lambda x: x[1], reverse=True)
        return blended[:k]


    def get_name(self) -> str:
        return f"Hybrid+Rerank({self.reranker.model_name.split('/')[-1]})"

In [33]:
# section 6.3
# hybrid_splade_dense = HybridRetriever([splade, dense])

reranked_retriever = RerankRetriever(
    retriever=hybrid_splade_dense,
    rerank_model="ncbi/MedCPT-Cross-Encoder",
    blend_strategy="position_aware",
    corpus=_corpus,
    retrieve_k=100
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
scores = evaluate_retriever(reranked_retriever, _queries, _relevant_docs)
for metric, value in scores.items():
    print(f"{metric}: {value:.4f}")

recall@1: 0.0367
recall@5: 0.1227
recall@10: 0.1701
recall@50: 0.3469
hit@1: 0.4660
hit@5: 0.6821
hit@10: 0.7407
hit@50: 0.8611
mrr: 0.5638


| Configuration | R@50       | H@50 | MRR | NDCG@10 |
|--------------|------------|------|-----|---------|
| **Hybrid (No Rerank)** | **0.3504** | 0.8580 | 0.5557 | — |
| + Generic CE (Rerank Only) | 0.2703     | 0.8364 | 0.5451 | — |
| + Medical CE (Rerank Only) | 0.3128     | 0.8519 | 0.5539 | — |
| + Weighted Blending (60% CE) | 0.3465     | **0.8611** | **0.5757** | — |
| + Position-Aware Blending | 0.3469     | 0.8611 | 0.5638 | — |

**Notes**
* Unlike bi-encoders (which encode query and document independently), cross-encoders process `(query, document)` pairs jointly through a transformer, enabling full attention between both texts. This captures nuanced relevance signals but is computationally expensive, making it suitable only for reranking a small candidate set.
* Score Normalization:
    - Cross-encoder logits →  Sigmoid: converts raw logits to (0,1) probabilities. Compresses extreme values while preserving rank order.
    - First-stage retriever scores → Min-Max: Linear scaling to [0,1]. Preserves relative distances for fair blending with CE scores
* Reranker-only approaches (even with medical domain CEs) underperform the hybrid baseline. Likely causes: (1) insufficient first-stage candidate recall limits what the reranker can improve, and (2) cross-encoders occasionally disagree with retrieval signals on top-ranked documents.
* Weighted blending (60% CE / 40% hybrid) achieves the best MRR while maintaining competitive recall. Position-aware blending shows promise and could surpass weighted blending with tuned rank weights or if first-stage ranking weaknesses are addressed. Hit rates improve consistently across all k, suggesting better ranking quality without sacrificing coverage.